In [1]:
import torch

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module
from tts.tokenizer.espeak_tokenizer import ESPEAKTokenizer

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/model_config.json"
aligner_training_module_ckpt_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/checkpoints_timit_bae/best_step_timit_bae_0.019927_step_266500_epoch_39.pth"

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"


txt_tokenizer = ESPEAKTokenizer()
model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(config=model_config).to(device=device)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cpu.
Loading nested state_dict from key 'model' in ./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/checkpoints_timit_bae/best_step_timit_bae_0.019927_step_266500_epoch_39.pth
⚠️ Checkpoint load summary (strict=False):
  - Unexpected top-level modules: ['vocoder']
Checkpoint loading process finished.


## INIT BenchMarkers

In [5]:
from tts.benchmark.timit.benchmarker import TIMITErrorAnalyzer
from tts.models.utils.input_maker import AlignerInputMaker

BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

input_maker = AlignerInputMaker(
    audio_config=model_config.nd_aligner.audio,
    preprocess_config=model_config.nd_aligner.preprocess,
    tokenizer_type=model_config.nd_aligner.tokenizer_type,
    zero_nonspeech_region=True,
    trim_nonspeech_region=True,
    device="cuda"
)

analyzer = TIMITErrorAnalyzer(
    # root_dir=TIMIT_ROOT,
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16000,
    hyp_audio_sr=22050,
    hyp_hop_length=256,
    input_maker=input_maker,
    analysis_dir="./results/timit_error_analysis",
    hyp_ignore_symbols=txt_tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
    seed=42,
)

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.
[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [6]:
rows = analyzer.analyze(
    aligner=aligner,
    vocoder=None,
    max_test_samples=None,
    top_k_alignments=50,
    compute_entropy=True,
    compute_mcd_dtw=False,
)

Saving worst alignments: 100%|██████████| 50/50 [01:06<00:00,  1.33s/it]
